<a href="https://colab.research.google.com/github/Shreya-Reddy-7/AI-Powered-Intelligent-Recruitment-Candidate-Analytics-System/blob/main/Resume_Parsing_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#DATASET


In [5]:
from google.colab import files
uploaded = files.upload()

Saving Resume.csv to Resume.csv


#PYTHON LIBARIES

In [7]:
!pip install spacy scikit-learn nltk
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 67.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [34]:
import pandas as pd
import re
import json
import spacy
import nltk

from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

In [35]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [36]:
nlp = spacy.load("en_core_web_sm")

# Load Resume Dataset (Extract Raw Text)

In [37]:
df = pd.read_csv("/content/Resume.csv")

df = df[['Resume_str']]   # only resume text
df.head()

,Resume_str
0,HR ADMINISTRATOR/MARKETING ASSOCIATE\...
1,"HR SPECIALIST, US HR OPERATIONS ..."
2,HR DIRECTOR Summary Over 2...
3,HR SPECIALIST Summary Dedica...
4,HR MANAGER Skill Highlights ...


#DATA CLEANING

In [38]:
def clean_text(text):

    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^A-Za-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)

    return text.lower()

df['cleaned_resume'] = df['Resume_str'].apply(clean_text)

df.head()

,Resume_str,cleaned_resume
0,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,hr administrator marketing associate hr admin...
1,"HR SPECIALIST, US HR OPERATIONS ...",hr specialist us hr operations summary versat...
2,HR DIRECTOR Summary Over 2...,hr director summary over years experience in ...
3,HR SPECIALIST Summary Dedica...,hr specialist summary dedicated driven and dy...
4,HR MANAGER Skill Highlights ...,hr manager skill highlights hr skills hr depa...


#Text Preprocessing (Tokenization, Stopword Removal, Lemmatization)

In [39]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [40]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):

    tokens = word_tokenize(text)

    tokens = [word for word in tokens if word not in stop_words]

    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    return " ".join(tokens)

df['processed_resume'] = df['cleaned_resume'].apply(preprocess_text)

In [41]:
df.head()

,Resume_str,cleaned_resume,processed_resume
0,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,hr administrator marketing associate hr admin...,hr administrator marketing associate hr admini...
1,"HR SPECIALIST, US HR OPERATIONS ...",hr specialist us hr operations summary versat...,hr specialist u hr operation summary versatile...
2,HR DIRECTOR Summary Over 2...,hr director summary over years experience in ...,hr director summary year experience recruiting...
3,HR SPECIALIST Summary Dedica...,hr specialist summary dedicated driven and dy...,hr specialist summary dedicated driven dynamic...
4,HR MANAGER Skill Highlights ...,hr manager skill highlights hr skills hr depa...,hr manager skill highlight hr skill hr departm...


#Extract Skills using TF-IDF

In [42]:
skills_list = [
'python','java','c++','sql','machine learning','deep learning','nlp',
'data analysis','data science','excel','power bi','tableau',
'html','css','javascript','react','node','flask','django',
'tensorflow','pytorch','scikit learn'
]

In [43]:
def extract_skills(text):

    skills_found = []

    for skill in skills_list:
        if skill in text:
            skills_found.append(skill)

    return list(set(skills_found))

In [44]:
df['skills'] = df['processed_resume'].apply(extract_skills)

#Extract Years of Experience and Education

In [45]:
def total_experience(text):

    text = text.lower()

    pattern = r'(\d+)\+?\s*(years|year|yrs|yr)'

    matches = re.findall(pattern, text)

    years = [int(match[0]) for match in matches]

    if years:
        return max(years)
    else:
        return 0

In [46]:
df['total_experience'] = df['Resume_str'].apply(total_experience)

In [47]:
education_keywords = [
'bachelor','bachelor of technology','b.tech','b.e',
'master','m.tech','mca','mba',
'bsc','msc','phd',
'computer science','information technology',
'electronics','mechanical engineering'
]

In [48]:
def extract_education(text):

    education_found = []

    for edu in education_keywords:
        if edu in text:
            education_found.append(edu)

    return list(set(education_found))

In [49]:
df['skills'] = df['skills'].apply(lambda x: x if x else ["Not Mentioned"])

In [50]:
df['education'] = df['cleaned_resume'].apply(extract_education)

In [51]:
df.head()

,Resume_str,cleaned_resume,processed_resume,skills,total_experience,education
0,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,hr administrator marketing associate hr admin...,hr administrator marketing associate hr admini...,[data analysis],15,[]
1,"HR SPECIALIST, US HR OPERATIONS ...",hr specialist us hr operations summary versat...,hr specialist u hr operation summary versatile...,[Not Mentioned],0,"[bachelor, master]"
2,HR DIRECTOR Summary Over 2...,hr director summary over years experience in ...,hr director summary year experience recruiting...,[excel],20,"[bachelor, mba, master]"
3,HR SPECIALIST Summary Dedica...,hr specialist summary dedicated driven and dy...,hr specialist summary dedicated driven dynamic...,[excel],20,[]
4,HR MANAGER Skill Highlights ...,hr manager skill highlights hr skills hr depa...,hr manager skill highlight hr skill hr departm...,[excel],0,[bachelor]


In [52]:
df.isnull().sum()#null values

,0
Resume_str,0
cleaned_resume,0
processed_resume,0
skills,0
total_experience,0
education,0


#Convert to JSON

In [53]:
import json

resume_json = []

for index,row in df.iterrows():

    data = {
        "resume_id": index + 1,
        "skills": row['skills'],
        "education": row['education'],
        "experience_years": row['total_experience']
    }

    resume_json.append(data)

json_output = json.dumps(resume_json, indent=4)

print(json_output)

[
    {
        "resume_id": 1,
        "skills": [
            "data analysis"
        ],
        "education": [],
        "experience_years": 15
    },
    {
        "resume_id": 2,
        "skills": [
            "Not Mentioned"
        ],
        "education": [
            "bachelor",
            "master"
        ],
        "experience_years": 0
    },
    {
        "resume_id": 3,
        "skills": [
            "excel"
        ],
        "education": [
            "bachelor",
            "mba",
            "master"
        ],
        "experience_years": 20
    },
    {
        "resume_id": 4,
        "skills": [
            "excel"
        ],
        "education": [],
        "experience_years": 20
    },
    {
        "resume_id": 5,
        "skills": [
            "excel"
        ],
        "education": [
            "bachelor"
        ],
        "experience_years": 0
    },
    {
        "resume_id": 6,
        "skills": [
            "excel"
        ],
        "education": [],

In [54]:
with open("parsed_resumes.json","w") as f:
    json.dump(resume_json,f,indent=4)

#Structured Resume Data in the Database

In [55]:
import sqlite3

In [56]:
conn = sqlite3.connect("resume_database.db")
cursor = conn.cursor()

In [57]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS resumes (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    skills TEXT,
    education TEXT,
    experience INTEGER
)
""")

In [58]:
import json

for index,row in df.iterrows():

    skills = json.dumps(row['skills'])
    education = json.dumps(row['education'])
    experience = row['total_experience']

    cursor.execute("""
    INSERT INTO resumes (skills, education, experience)
    VALUES (?, ?, ?)
    """,(skills, education, experience))

In [59]:
conn.commit()

In [60]:
cursor.execute("SELECT * FROM resumes LIMIT 5")

rows = cursor.fetchall()

for r in rows:
    print(r)

(1, '["data analysis"]', '[]', 15)
(2, '["Not Mentioned"]', '["bachelor", "master"]', 0)
(3, '["excel"]', '["bachelor", "mba", "master"]', 20)
(4, '["excel"]', '[]', 20)
(5, '["excel"]', '["bachelor"]', 0)


In [61]:
conn.close()